In [12]:
# This test program aims to calculate the attenuation factors using 1000 MC sample points and for one .nxspe file.
# This program works for single crystals.

using FileIO
using GeometryBasics
using LinearAlgebra
using BenchmarkTools
using MeshIO
using SparseArrays
using StaticArrays
using LoopVectorization
include("Modules/sampling.jl")
using .sampling
include("Modules/nxspe.jl")
using .nxspe
include("Modules/sin_crystal.jl")
using .sin_crystal

In [2]:
# Extracting the contents of the .nxspe file.

en_i, azi, pol, data, Δen = nxspe.extract("test_nxspe_data/LET104215_3.7meV_1to1.nxspe")

(3.7f0, Float32[-137.16072, -137.39026, -137.62152, -137.85446, -138.08911, -138.32544, -138.5635, -138.80327, -139.04478, -139.28812  …  41.245438, 41.48468, 41.72211, 41.957756, 42.191696, 42.42389, 42.654366, 42.883137, 43.110214, 43.335594], Float32[48.282505, 48.177177, 48.07187, 47.966606, 47.861397, 47.756275, 47.65122, 47.54627, 47.441406, 47.336624  …  131.69145, 131.58684, 131.4822, 131.3775, 131.27277, 131.16801, 131.06323, 130.95845, 130.85367, 130.74889], Float32[NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], Float32[-2.96, -2.9415, -2.923, -2.9045, -2.886, -2.8675, -2.849, -2.8305, -2.812, -2.7935  …  2.7935, 2.812, 2.8305, 2.849, 2.8675, 2.886, 2.9045, 2.923, 2.9415, 2.96])

In [4]:
# Calculating ki, n_bins, n_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = nxspe.magk_calc(en_i)
# Converting ki to a static array.
ki = SVector{3, Float32}(ki)

# Extracting the number of energy bins and number of detectors.
const n_bins = length(Δen) - 1
const n_detectors = length(azi)

98304

In [5]:
# Calculating the final neutron energy, in meV, for each energy bin.

ef_bins = nxspe.ef_calc(en_i, Δen, n_bins)

320-element Vector{Float32}:
 6.65075
 6.63225
 6.6137505
 6.59525
 6.57675
 6.5582504
 6.53975
 6.52125
 6.5027504
 6.48425
 ⋮
 0.89724994
 0.8787501
 0.86025023
 0.8417499
 0.82325006
 0.8047502
 0.7862499
 0.76775
 0.7492502

In [6]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.

kx, ky, kz = nxspe.kf_calc(ef_bins, pol, azi)

(Float32[-0.98057234 -0.9825924 … 0.9892723 0.987179; -0.9792076 -0.98122483 … 0.9878954 0.98580503; … ; -0.3331611 -0.3338474 … 0.336117 0.33540577; -0.32912263 -0.32980064 … 0.33204272 0.3313401], Float32[-0.9092696 -0.9038486 … 0.9260755 0.93142897; -0.90800405 -0.9025906 … 0.9247866 0.93013257; … ; -0.30893514 -0.30709326 … 0.31464514 0.31646404; -0.30519032 -0.3033708 … 0.31083113 0.31262797], Float32[1.1921976 1.1946537 … -1.1719011 -1.1694212; 1.1905382 1.192991 … -1.1702701 -1.1677935; … ; 0.40506324 0.40589777 … -0.3981673 -0.3973247; 0.40015322 0.40097764 … -0.39334086 -0.39250848])

In [7]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


stl = load("crystal.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

2142

In [8]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s, e2s, e3s = sampling.ve_calc(vertices, indices)

(SVector{3, Float32}[[13.304199, 20.258078, 46.79252], [13.590828, 20.380196, 46.776127], [13.963831, 20.172203, 46.80407], [13.192872, 20.479755, 46.77794], [13.428433, 20.525827, 46.78789], [13.832438, 20.393225, 46.786858], [12.922315, 20.37644, 46.789036], [13.590828, 20.380196, 46.776127], [13.192872, 20.479755, 46.77794], [13.025568, 20.651573, 46.79539]  …  [14.65111, 20.228249, 49.119], [14.667707, 20.486109, 49.11232], [15.205641, 20.177717, 49.115513], [15.644693, 20.509438, 49.081673], [14.14268, 20.198109, 49.116863], [14.123109, 20.603514, 49.111305], [14.391234, 20.406094, 49.118073], [14.956343, 20.40861, 49.111523], [14.391234, 20.406094, 49.118073], [13.824862, 20.41183, 49.098145]], SVector{3, Float32}[[0.28662872, 0.122117996, -0.016391754], [0.24161053, 0.0130290985, 0.010730743], [-0.45956707, -0.039129257, 0.010292053], [0.23556137, 0.046072006, 0.009952545], [0.16239452, -0.14563179, -0.011764526], [0.25931644, 0.02025795, 0.012813568], [-0.32102966, -0.09597397,

In [9]:
# Setting the desired number of MC sample points and creating vector for coordinates.

const n_mc = 10
mc_coords = Vector{SVector{3, Float32}}(undef, n_mc)

10-element Vector{SVector{3, Float32}}:
 [0.0, 0.0, 1.0f-44]
 [1.1f-44, 3.5f-44, 3.4f-44]
 [3.6f-44, 4.2f-44, 4.3f-44]
 [4.6f-44, 5.5f-44, 5.6f-44]
 [7.7f-44, 7.7f-44, 7.7f-44]
 [7.1f-44, 7.1f-44, 7.1f-44]
 [0.0, 6.7f-44, 0.0]
 [0.0, 0.0, 0.0]
 [6.0f-44, 0.0, 5.2f-44]
 [5.3f-44, 7.0f-45, 0.0]

In [10]:
# Setting the (estimated) parameters of the sample.

# The reference attenuation coefficent at 25.3 meV in cm^-1.
const μ_ref = Float32(1)
const en_ref = Float32(25.3)

25.3f0

In [13]:
# Calculating the pre-scattering absorption cross section.

const μi = sin_crystal.μ_calc(μ_ref, en_ref, en_i)

2.6149259f0

In [ ]:
# Defining the function to determine the length of the paths the neutrons take within the sample.

"""
Calculates the distance between a given point in the sample (origin) and a triangular face of the mesh that describes the surface. 
Iterates through each face to determine which one is intersected.
Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

Parameters
----------
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
d (3-vector with float elements): Normalised direction vector.
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
origin (3-vector with float elements): Coordinates of scattering sites.
v1s (n_faces-vector of 3-vectors with float elements): First vertex of each face, V1.

Returns
-------
path_length (float): Distance between origin and the surface the neutron path intersects, in units of the .stl file. nothing is outputted if there is no intersection.
"""
function len_calc(
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}}, 
    d :: SVector{3, Float32}, 
    ps :: Vector{SVector{3, Float32}}, 
    dets :: Vector{Float32}, 
    origin :: SVector{3, Float32}, 
    v1s :: Vector{SVector{3, Float32}}
    ) :: Union{Float32, Nothing}
    # Iterating through all faces.
    @inbounds for j in 1:n_faces
        # If det = p.e2 = (d x e3).e2 = 0, the path is parallel to the triangular face, so it can never intersect it.
        # Keeping only negative determinants, culling front-facing triangles as we are inside the mesh.
        det = dets[j]
        if det < -1f-6
            # Pre-computing the inverse determinant.
            inv_det = 1 / det
            # Calculating t = origin - V1 and q = t x e2 required for the MT algorithm.
            t = origin - v1s[j]
            q = cross(t, e2s[j])
            # Calculating the barycentric coordinates, (u,v), of the intersection.
            u = inv_det * (dot(ps[j], t))
            v = inv_det * (dot(q, d))
            # Determining whether the intersection point lies within the triangle.
            if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
                # Neutron's path described by r(λ) = origin + λd.
                λ = inv_det * dot(q, e3s[j])
                # Only accepting positive λ as this indicates paths moving in positive direction of d.
                if λ > 0
                    # The path length is simply λ as the direction vector is normalised.
                    return Float32(λ)
                end
            end
        end
    end
    # Returning nothing if no path is intersected.
    # Assuming the origin is within the sample, then this only occurs due to floating point precision errors.
    # ie u+v = 1.00000001 > 1.
    return nothing
end


#test = mc_coords[1]
#@benchmark len_calc(e2s, e3s, di, p_i, det_i, test, v1s)

In [ ]:
# Defining a function to pre-calculate p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

"""
Calculates p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

Parameters
----------
d (3-vector with float elements): Normalised direction vector.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
ps (n_faces-vector of 3-vectors with float elements): Pre-allocated vector.
dets (n_faces-vector with float elements): Pre-allocated vector.

Returns
-------
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
"""
function pdet_calc!(
    d :: SVector{3, Float32}, 
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}},
    ps :: Vector{SVector{3, Float32}},
    dets :: Vector{Float32} 
    ) :: Tuple{Vector{SVector{3, Float32}}, Vector{Float32}}
    # @inbounds is used to remove checks on the index i as we are sure of the sizes of our arrays.
    # @simd is used to vectorize and speed up the loop.
    @inbounds @simd for i in 1:n_faces
        # Calculating cross products, p = d x e3, for the direction vector, d, and for each face.
        ps[i] = cross(d, e3s[i])
        # Calculating the determinant = p.e2 = (d x e3).e2 for the direction vector, d, and for each face.
        dets[i] = dot(ps[i], e2s[i])
    end
    return ps, dets
end

# @benchmark pdet_calc!(di, e2s, e3s, p_i, det_i)

In [14]:
# Generating the sample points.

# Calculating the extrema of the axis-aligned bounding box around the sample.
ranges = sampling.aabb_3d(vertices)
sampling.sample!(ranges, e2s, e3s, v1s, mc_coords, n_faces, n_mc)

10-element Vector{SVector{3, Float32}}:
 [18.126554, 19.667877, 48.049263]
 [18.161755, 18.99024, 48.80512]
 [19.192299, 18.71793, 48.25516]
 [16.481798, 20.60404, 47.386944]
 [13.569286, 19.808813, 48.59226]
 [14.614704, 21.903175, 48.017857]
 [12.951132, 20.149704, 48.55248]
 [12.85615, 20.543264, 48.73087]
 [16.083738, 19.513659, 48.229347]
 [13.968847, 21.128551, 47.042587]

In [ ]:
# The pre-scattering neutron path lengths are dependent only on the MC coordinates.
# They can, therefore, be calculated and stored.

# Normalising the pre-scattering direction vector.
di = -ki / norm(-ki)
# Pre-allocating these vectors.
p_i = Vector{SVector{3, Float32}}(undef, n_faces)
det_i = Vector{Float32}(undef, n_faces)
# Calculating p and det required for the MT algorithm.
pdet_calc!(di, e2s, e3s, p_i, det_i)
len_i = Float32.(zeros(n_mc))
# Iterating through the Monte Carlo sample points to find the pre-scattering path length of each.
for i in 1:n_mc
    len_i[i] = len_calc(e2s, e3s, di, p_i, det_i, mc_coords[i], v1s)
end

In [ ]:
# Defining the function that will calculate the attenuation factor given a certain energy bin and wavevector.

"""
Calculates the attenuation factor given a set initial and final energy and wavevector.

Parameters
----------
df (3-vector with float elements): Normalised post-scattering neutron direction vector, in Angstrom^-1.
en_f (float): Post-scattering energy of neutron, in meV.
v1s (n_faces-vector of 3-vectors with float elements): First vertex of each face, V1.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector of float elements): Pre-scattering path length of neutron, in units of .stl file.
p_f (n_faces-vector of 3-vectors with float elements): Pre-allocated vector.
det_f (n_faces-vector with float elements): Pre-allocated vector.

Returns
-------
atten_calc (float): Attenuation factor.
"""
function atten_calc(
    df :: SVector{3, Float32}, 
    en_f :: Float32, 
    v1s :: Vector{SVector{3, Float32}}, 
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}}, 
    coords :: Vector{SVector{3, Float32}}, 
    len_i :: Vector{Float32},
    p_f :: Vector{SVector{3, Float32}},
    det_f :: Vector{Float32}
    ) :: Float32
    # Calculating the absorption cross section after the neutron scatters.
    axsf = axs_calc(en_f)
    # Setting up the Moller-Trumbore algorithm for ray-triangle intersections.
    pdet_calc!(df, e2s, e3s, p_f, det_f)
    # Tallying the number of accepted MC sample points where a path length could be calculated.
    acc_pts = 0
    atten = 0
    @inbounds for i in 1:n_mc
        # Calculating the path length, len_f, at this sample point.
        len_f = len_calc(e2s, e3s, df, p_f, det_f, coords[i], v1s)
        if typeof(len_f) == Float32
            # Adding the attenuation factor contribution from this sample point to A.
            atten += exp(-n * axsi * len_i[i]) * exp(-n * axsf * len_f)
            acc_pts += 1
        end
    end
    # Dividing by the total number of contributing sample points.
    if acc_pts == 0
        error("The path length could not be calculated for any sample point. Are they all within the sample?")
    else
        atten = atten / (acc_pts)
        return atten
    end
end

# df_test = SVector{3}(kx[1,1], ky[1,1], kz[1,1])
# df_test = df_test / norm(df_test)
# @benchmark atten_calc(df_test, ef_bins[1], v1s, e2s, e3s, mc_coords, len_i, p_i, det_i)

In [ ]:
# Converting the grid of data and attenuation factors to sparse arrays.

# Replacing every NaN value with zero to allow conversion to sparse matrix.
data_copy = copy(data)
data_copy .= ifelse.(isnan.(data_copy), 0, data)
s_data = sparse(data_copy)

In [ ]:
# Defining the function that calculates the grid of attenuation factors.

"""
Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.

Parameters
----------
s_data (n_bins x n_detectors sparse matrix with float elements): Neutron signal measured at different detectors for different energy bins.
kx (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
en_i (float): Pre-scattering neutron energy, in meV.
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
v1s (n_faces-vector of 3-vectors with float elements): First vertex of each face, V1.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector with float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
atten_grid (n_bins x n_detectors matrix with float elements): Attenuation factor for different detectors and energy bins.
"""
function a_grid_calc(
    s_data :: SparseMatrixCSC{Float32, Int64}, 
    kx :: AbstractMatrix{Float32}, 
    ky :: AbstractMatrix{Float32}, 
    kz :: AbstractMatrix{Float32}, 
    ef_bins :: SVector{n_bins, Float32}, 
    v1s :: Vector{SVector{3, Float32}}, 
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}}, 
    coords :: Vector{SVector{3, Float32}}, 
    len_i :: Vector{Float32}
    ) :: Matrix{Float32}
    atten_grid = zeros(Float32, n_bins, n_detectors)
    # Determining the locations in which the signal is either NaN or 0 as we don't want to calculate atten there.
    idx = findnz(s_data)
    # Pre-allocating the vectors needed for the MT algorithm.
    p_f = Vector{SVector{3, Float32}}(undef, n_faces)
    det_f = Vector{Float32}(undef, n_faces)
    # Skipping checks on array lengths using @inbounds.
    @inbounds for (i, j) in zip(idx[1], idx[2])
        kf = SVector{3, Float32}(kx[i, j], ky[i, j], kz[i, j])
        # Normalising the post-scattering direction vector.
        df = kf / norm(kf)
        atten_grid[i, j] = atten_calc(df, ef_bins[i], v1s, e2s, e3s, coords, len_i, p_f, det_f)
    end
    return atten_grid
end

# @benchmark a_grid_calc(s_data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i)

In [ ]:
# Testing the time taken to output this grid of attenuation factors.

atten_grid = a_grid_calc(s_data, kx, ky, kz, ef_bins, v1s, e2s, e3s, mc_coords, len_i)
# Testing the same (known to be non-zero) datapoint.
display(atten_grid[160,6])
# Converting the attenuation factors to a sparse matrix.
s_atten = sparse(atten_grid)
display(s_atten)